In [84]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn .pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score


In [85]:
#load the data set 
housing = pd.read_csv("housing.csv")
housing.head()



,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [86]:
# craete the statified tset set 
housing["income_cat"]=pd .cut (housing["median_income"], bins=[0., 1.5, 3.0, 4.5, 6., np.inf], labels=[1, 2, 3, 4, 5]   )
housing 

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,income_cat
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,5
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,5
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,5
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,4
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,3
...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND,2
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND,2
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND,2
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND,2


In [87]:
split =StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]
    housing = strat_train_set.copy()

In [88]:
#spearte the feature and labels 
housing_labels= housing["median_house_value"].copy()
housing=housing.drop("median_house_value", axis=1)
print(housing_labels)


12655     72100.0
15502    279600.0
2908      82700.0
14053    112500.0
20496    238300.0
           ...   
15174    268500.0
12661     90400.0
19263    140400.0
19140    258100.0
19773     62700.0
Name: median_house_value, Length: 16512, dtype: float64


In [89]:
#4. spearte the numerical and catergorial column 
num_attribute = housing.drop(["ocean_proximity"], axis=1).columns.tolist()
cat_attribute =["ocean_proximity"]
print(num_attribute)
print(cat_attribute)


['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'income_cat']
['ocean_proximity']


In [90]:
#build the pipeline for numeriacal and catergorial data
num_pipeline = Pipeline([("imputer" , SimpleImputer(strategy="median")), ("std_scaler", StandardScaler())])
cat_pipeline= Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))])

In [91]:
full_pipeline=ColumnTransformer([("num", num_pipeline, num_attribute), ("cat", cat_pipeline, cat_attribute)])
#transform the data 
housing_pr = full_pipeline.fit_transform(housing)
print(housing_pr)

[[-0.94135046  1.34743822  0.02756357 ...  0.          0.
   0.        ]
 [ 1.17178212 -1.19243966 -1.72201763 ...  0.          0.
   1.        ]
 [ 0.26758118 -0.1259716   1.22045984 ...  0.          0.
   0.        ]
 ...
 [-1.5707942   1.31001828  1.53856552 ...  0.          0.
   0.        ]
 [-1.56080303  1.2492109  -1.1653327  ...  0.          0.
   0.        ]
 [-1.28105026  2.02567448 -0.13148926 ...  0.          0.
   0.        ]]


In [92]:
lin_reg= LinearRegression()
lin_reg.fit(housing_pr, housing_labels)
lin_preds=lin_reg.predict(housing_pr)
lin_rmse = root_mean_squared_error(housing_labels, lin_preds)
print(lin_rmse)
lin_cv_scores = -cross_val_score(lin_reg, housing_pr, housing_labels, scoring="neg_mean_squared_error", cv=10)
print(lin_cv_scores)


68866.78550087014
[5.19815429e+09 4.25929230e+09 4.55284050e+09 4.80863798e+09
 4.40965518e+09 5.29454947e+09 4.93808582e+09 4.80621616e+09
 4.44554068e+09 4.98218347e+09]


In [93]:
#decision tree
from sklearn.tree import DecisionTreeRegressor
tree_reg = DecisionTreeRegressor()
tree_reg.fit(housing_pr, housing_labels)
tree_preds = tree_reg.predict(housing_pr)
tree_rmse = root_mean_squared_error(housing_labels, tree_preds)
print(tree_rmse)# it si 0 so it is overfitting the data
tree_cv_scores = -cross_val_score(tree_reg, housing_pr, housing_labels, scoring="neg_mean_squared_error", cv=10)
print(tree_cv_scores)

0.0
[5.05651097e+09 4.80024635e+09 4.29121440e+09 4.88007527e+09
 4.58766742e+09 4.51017740e+09 5.48178846e+09 4.96320337e+09
 4.61482281e+09 5.28425678e+09]


In [95]:
##random forest
from sklearn.ensemble import RandomForestRegressor
forest_reg = RandomForestRegressor()
forest_reg.fit(housing_pr, housing_labels)
forest_preds = forest_reg.predict(housing_pr)
forest_rmse = root_mean_squared_error(housing_labels, forest_preds)
print(forest_rmse)
forest_cv_scores = -cross_val_score(forest_reg, housing_pr, housing_labels, scoring="neg_mean_squared_error", cv=10)
print(forest_cv_scores)

18442.983768539
[2.53601347e+09 2.41319018e+09 2.13305842e+09 2.57320243e+09
 2.26178410e+09 2.40583592e+09 2.61089038e+09 2.41784610e+09
 2.25705702e+09 2.82284104e+09]


In [96]:
#joblib it is use for the saving the model , beacuse i can not train the model again and again 

In [ ]:
i